# 条件训练与数字生成

In [ ]:
from pathlib import Path
import os, sys, torch
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'bs6221_flow').is_dir())
sys.path.insert(0, str(ROOT))
os.environ.setdefault('MPLCONFIGDIR', str(ROOT / '.cache/matplotlib'))
torch.set_num_threads(4)
from bs6221_flow.sampling import load_flow, generate_images, compare_solvers, solve_ode
from bs6221_flow.training import train_model
from bs6221_flow.presentation import show_training, forward_demo, probability_paths, generation_demo, diffusion_demo, show_saved_numerics
from bs6221_flow.widgets import generation_panel, numerics_panel
from IPython.display import display
DEVICE = os.environ.get('BS6221_DEVICE', 'auto')
FLOW_WEIGHTS = None  # None: frozen 15,000-step EMA; or your training run's best.pt
model = load_flow(FLOW_WEIGHTS, DEVICE)
print('Device:', next(model.parameters()).device, '| weights SHA-256:', model.flow_weight_sha256)

## 环境与可选训练
先按 README 创建环境。默认直接加载预训练模型，下面不会自动训练。
历史训练曲线来自保存的 checkpoint；Flow 的速度 MSE 与 DDPM 的噪声 MSE 不可直接比较。
历史训练有放回抽样，图中 epoch 是等效遍历次数；新训练按随机打乱后的完整数据遍历。
续训的验证协议与早期不同，分开画线；没有记录的训练损失保持缺失。

In [ ]:
show_training(kind='flow')
show_training(kind='diffusion')

In [ ]:
RUN_TRAINING = os.environ.get('BS6221_RUN_TRAINING', '0') == '1'
TRAIN_CONFIG = dict(kind='flow', steps=1000, batch_size=64, learning_rate=3e-4,
                    ema_decay=.995, seed=42, validation_every=250,
                    validation_samples=256, device_preference=DEVICE)
# To train full epochs: remove steps and add epochs=1. Each epoch covers 55,000 images.
print('Optional training parameters:', TRAIN_CONFIG)
if RUN_TRAINING:
    training_folder, history = train_model(**TRAIN_CONFIG)
    show_training(history=history, kind=TRAIN_CONFIG['kind'])
    if TRAIN_CONFIG['kind'] == 'flow':
        model = load_flow(training_folder / 'best.pt', DEVICE)

## 2. 选择数字和噪声，再看生成过程
选择数字、噪声种子、求解器和步数。先预览噪声，再生成图片；相同模型、标签和初始噪声可重复。
相同 NFE 比较会使用同一噪声；改变噪声尺度属于扰动实验，默认标准高斯尺度为 1。
实际路径来自积分过程，不是图片插值。可调用 generate_images(..., noise=你的张量, net=model)。

In [ ]:
playground = generation_panel(model)
display(playground)